In [1]:
# Figure 2: Summary of test results

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 

from fast_o3_iv import metrics, utils
from fast_o3_iv.config import PROCESSED_DATA_DIR


In [ ]:
MODEL_KEYS = [
    "MLR-global",
    "XGB-global",
    "GPR-local",
    "MLR-local",
    "XGB-local",
]


def load_test_results(model_key):
    """Load and concatenate all test-set prediction results for a given model_key.
    """

    TEST_DIR = PROCESSED_DATA_DIR / "model_outputs" / "testing"
    matching_files = sorted(TEST_DIR.glob(f"*.{model_key}.parquet"))

    dfs = []
    for file_path in matching_files:
        parts = file_path.stem.split(".")
        ensemble_member = ".".join(parts[2:4])  # e.g. "wa6_ic3.002"

        df = pd.read_parquet(file_path)
        df["model_key"] = model_key
        df["ensemble_member"] = ensemble_member
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [4]:
TEST_METRICS = ["ens_clim", "ens_trend", "resid_sd", "resid_q90", "pgt_q90", "emd_to_noise"]

results = {}

for model_key in MODEL_KEYS:
    df = load_test_results(model_key)


    summary = metrics.calculate_global_test_metrics(df, only_populated_grid_cells=True)

    results[model_key] = {
        metric: np.round(float(summary[metric]), 2) for metric in TEST_METRICS
    }

table = pd.DataFrame(results).reindex(TEST_METRICS)

print(table)

table.to_csv(PROCESSED_DATA_DIR / "figure_data" / "Table2_test_metrics.csv")

/home/eleroy/proj-dirs/fast-o3-iv/.venv/lib/python3.11/site-packages/xarray/groupers.py:543: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/home/eleroy/proj-dirs/fast-o3-iv/src/fast_o3_iv/metrics.py:221: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return {name: ds.merge(aux) for name, ds in fields.items()}
/home/eleroy/proj-dirs/fast-o3-iv/src/fast_o3_iv/metrics.py:221: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping

              MLR-global  XGB-global  GPR-local  MLR-local  XGB-local
ens_clim            2.22        0.28       0.14       0.14       0.14
ens_trend           6.06        0.83       0.49       0.46       0.49
resid_sd            1.96        0.95       0.66       0.87       0.88
resid_q90           2.25        1.10       0.67       1.01       0.98
pgt_q90             0.10        0.06       0.04       0.06       0.06
emd_to_noise        0.49        0.24       0.16       0.22       0.24
